<a href="https://colab.research.google.com/github/valsson-group/UNT-ChemicalApplicationsOfMachineLearning-Spring2026/blob/main/Lecture-16-March-24-2026/Lecture_16_DimensionalityReduction_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Lecture 16 - Dimensionality Reduction 2


Import all basic pacakges

In [ ]:
# basic
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
!pip install opentsne

## Alanine Dipeptide

Here we consider a alanine dipeptide in vacuum that is simple model often used to test methods. It consists of one alanine (ala) residues and capping group. It is conformational dynamics can be understood in terms of the backbond dihedral angles $\Phi$ and $\Psi$.

![Alanine Dipeptide](https://github.com/valsson-group/UNT-ChemicalApplicationsOfMachineLearning-Spring2026/blob/main/Lecture-16-March-24-2026/Ala1.png?raw=true)

We will consider dataset from molecular dynamics simulations, in particular parallel tempering (PT) simulations, that should give correct equilibrium sampling according to the Boltzmann distribution.

The dataset we consider is at different temperatures, 416 K and 576 K. It is from a 100 ns PT simulations where we save variables every 10 ps, so we have 10000 samples.

The variables we consider are the dihedrals, $\Phi$ and $\Psi$, and also the set of all heavy atom distances.

In [ ]:
# Download datasets

%%bash
datasets="
AlanineDipeptide_T-416K_Dihedrals.data
AlanineDipeptide_T-416K_HeavyAtomDistances.data
AlanineDipeptide_T-576K_Dihedrals.data
AlanineDipeptide_T-576K_HeavyAtomDistances.data
"

url="https://raw.githubusercontent.com/valsson-group/UNT-ChemicalApplicationsOfMachineLearning-Spring2026/refs/heads/main/Lecture-16_March-24-2026/Datasets"

for d in ${datasets}
do
  wget ${url}/${d} &> /dev/null
done

ls

First we consider the dihedral angles

In [ ]:
!head AlanineDipeptide_T-416K_Dihedrals.data

In [ ]:
def get_variables_names_from_header(filename):
  with open(filename, 'r') as f:
    header = f.readline()
    variables = header.split()[2:]
  return variables

In [ ]:
dih_variables = get_variables_names_from_header("AlanineDipeptide_T-416K_Dihedrals.data")

data_dih = pd.read_csv("AlanineDipeptide_T-416K_Dihedrals.data", header=None, names=dih_variables, sep='\\s+', comment="#")

# only take every N-th value
stride = 1
data_dih = data_dih.iloc[::stride]
# this is to reset the index
data_dih.reset_index(drop=True, inplace=True)

In [ ]:
data_dih

In [ ]:
x_label = 'phi'
y_label = 'psi'

t = data_dih['time']
x = data_dih[x_label]
y = data_dih[y_label]


plt.plot(t, x, '.', label='x')
plt.xlabel('time')
plt.ylabel(x_label)
plt.show()

plt.plot(t, y, '.', label='y')
plt.xlabel('time')
plt.ylabel(y_label)
plt.show()

g = sns.JointGrid(height=7, ratio=5)
ax_scatter, ax_top, ax_right = g.ax_joint, g.ax_marg_x, g.ax_marg_y
sc = ax_scatter.scatter(x, y, s=14, alpha=0.8, linewidths=0)
ax_scatter.set_xlabel(x_label)
ax_scatter.set_ylabel(y_label)
sns.kdeplot(x=x, ax=ax_top, fill=True, color="#a855f7", bw_adjust=0.6)
sns.kdeplot(y=y, ax=ax_right, fill=True, color="#f97316", bw_adjust=0.6)
plt.show()

g = sns.JointGrid(height=7, ratio=5)
ax_joint, ax_top, ax_right = g.ax_joint, g.ax_marg_x, g.ax_marg_y
sns.kdeplot(x=x, y=y,
            ax=ax_joint,
            bw_adjust=0.6,
            fill=True,
            levels=8)
ax_joint.set_xlabel(x_label)
ax_joint.set_ylabel(y_label)
sns.kdeplot(x=x, ax=ax_top, fill=True, color="#a855f7", bw_adjust=0.6)
sns.kdeplot(y=y, ax=ax_right, fill=True, color="#f97316", bw_adjust=0.6)
plt.show()



In [ ]:
# KDE with correct treatment of periodic variables by replicating
# the data

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

stride=1

data = np.vstack([data_dih['phi'],data_dih['psi']])

periodic_variables = True

if periodic_variables:
  # Tile data in both dimensions
  shifts = [-2*np.pi, 0, 2*np.pi]
  tiled_data = []
  for dx in shifts:
      for dy in shifts:
          shifted = np.vstack([data_dih['phi'] + dx, data_dih['psi'] + dy])
          tiled_data.append(shifted)

  data_used = np.hstack(tiled_data)
else:
  data_used = data

data_used = data_used[:,::stride]

x = data_used[0]
y = data_used[1]

g = sns.JointGrid(height=7, ratio=5)
ax_joint, ax_top, ax_right = g.ax_joint, g.ax_marg_x, g.ax_marg_y
sns.kdeplot(x=x, y=y,
            ax=ax_joint,
            bw_adjust=0.6,
            fill=True,
            levels=8)
ax_joint.set_xlabel(x_label)
ax_joint.set_ylabel(y_label)
sns.kdeplot(x=x, ax=ax_top, fill=True, color="#a855f7", bw_adjust=0.6)
sns.kdeplot(y=y, ax=ax_right, fill=True, color="#f97316", bw_adjust=0.6)
ax_joint.set_xlim(-np.pi, np.pi)
ax_joint.set_ylim(-np.pi, np.pi)
ax_top.set_xlim(-np.pi, np.pi)
ax_right.set_ylim(-np.pi, np.pi)
plt.show()


## PCA

Let first do PCA on the cosine and sine of the dihedral angles

In [ ]:
data_dih['cos_phi'] = np.cos(data_dih['phi'])
data_dih['sin_phi'] = np.sin(data_dih['phi'])
data_dih['cos_psi'] = np.cos(data_dih['psi'])
data_dih['sin_psi'] = np.sin(data_dih['psi'])

In [ ]:
from sklearn.decomposition import PCA

features = data_dih[ ['cos_phi',
                      'sin_phi',
                      'cos_psi',
                      'sin_psi'] ]

# Apply PCA to 2D features
pca = PCA()
pca.fit(features)

print("Explained variance ratio for 2D features:")
display(pca.explained_variance_ratio_)
print("Cumulative explained variance for 2D features:")
display(pca.explained_variance_ratio_.cumsum())

features_pca = pca.transform(features)
x=features_pca[:,0]
y=features_pca[:,1]

plt.scatter(x, y, s=14, alpha=0.8, linewidths=0, c=data_dih['phi'])
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.colorbar()
plt.show()

g = sns.JointGrid(height=7, ratio=5)
ax_scatter, ax_top, ax_right = g.ax_joint, g.ax_marg_x, g.ax_marg_y
sc = ax_scatter.scatter(x, y, s=14, alpha=0.8, linewidths=0, c=data_dih['phi'])
ax_scatter.set_xlabel("PC 1")
ax_scatter.set_ylabel("PC 2")
sns.kdeplot(x=x, ax=ax_top, fill=True, color="#a855f7", bw_adjust=0.6)
sns.kdeplot(y=y, ax=ax_right, fill=True, color="#f97316", bw_adjust=0.6)
plt.show()

g = sns.JointGrid(height=7, ratio=5)
ax_joint, ax_top, ax_right = g.ax_joint, g.ax_marg_x, g.ax_marg_y
sns.kdeplot(x=x, y=y,
            ax=ax_joint,
            bw_adjust=0.6,
            fill=True,
            levels=8)
ax_joint.set_xlabel("PC 1")
ax_joint.set_ylabel("PC 2")
sns.kdeplot(x=x, ax=ax_top, fill=True, color="#a855f7", bw_adjust=0.6)
sns.kdeplot(y=y, ax=ax_right, fill=True, color="#f97316", bw_adjust=0.6)
plt.show()


In [ ]:
!head AlanineDipeptide_T-416K_HeavyAtomDistances.data

In [ ]:
def get_variables_names_from_header(filename):
  with open(filename, 'r') as f:
    header = f.readline()
    variables = header.split()[2:]
  return variables

In [ ]:
dist_variables = get_variables_names_from_header("AlanineDipeptide_T-416K_HeavyAtomDistances.data")

In [ ]:
dist_variables

In [ ]:
data_dist = pd.read_csv("AlanineDipeptide_T-416K_HeavyAtomDistances.data", header=None, names=dist_variables, sep='\\s+', comment="#")

# only take every N-th value
stride = 1
data_dist = data_dist.iloc[::stride]
# this is to reset the index
data_dist.reset_index(drop=True, inplace=True)

In [ ]:
from sklearn.decomposition import PCA

features = data_dist.drop(columns=['time'])

# Apply PCA to 2D features
pca = PCA()
pca.fit(features)

print("Explained variance ratio for 2D features:")
display(pca.explained_variance_ratio_)
print("Cumulative explained variance for 2D features:")
display(pca.explained_variance_ratio_.cumsum())

features_pca = pca.transform(features)
x=features_pca[:,0]
y=features_pca[:,1]

plt.scatter(x, y, s=14, alpha=0.8, linewidths=0, c=data_dih['phi'])
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.colorbar()
plt.show()

g = sns.JointGrid(height=7, ratio=5)
ax_scatter, ax_top, ax_right = g.ax_joint, g.ax_marg_x, g.ax_marg_y
sc = ax_scatter.scatter(x, y, s=14, alpha=0.8, linewidths=0, c=data_dih['phi'])
ax_scatter.set_xlabel("PC 1")
ax_scatter.set_ylabel("PC 2")
sns.kdeplot(x=x, ax=ax_top, fill=True, color="#a855f7", bw_adjust=0.6)
sns.kdeplot(y=y, ax=ax_right, fill=True, color="#f97316", bw_adjust=0.6)
plt.show()

g = sns.JointGrid(height=7, ratio=5)
ax_joint, ax_top, ax_right = g.ax_joint, g.ax_marg_x, g.ax_marg_y
sns.kdeplot(x=x, y=y,
            ax=ax_joint,
            bw_adjust=0.6,
            fill=True,
            levels=8)
ax_joint.set_xlabel("PC 1")
ax_joint.set_ylabel("PC 2")
sns.kdeplot(x=x, ax=ax_top, fill=True, color="#a855f7", bw_adjust=0.6)
sns.kdeplot(y=y, ax=ax_right, fill=True, color="#f97316", bw_adjust=0.6)
plt.show()


## *t*-SNE

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

features =

phi = data_dih['phi']

# only take every N-th value
stride = 10

features = features.iloc[::stride]
features.reset_index(drop=True, inplace=True)
print(features.shape)

phi = phi.iloc[::stride]
phi.reset_index(drop=True, inplace=True)
print(phi.shape)




perplexities = [5, 10, 20, 40, 80, 160]

for perplexity in perplexities:
  tsne = TSNE(perplexity=perplexity)
  emb = tsne.fit_transform(features)

  x = emb[:,0]
  y = emb[:,1]

  g = sns.JointGrid(height=7, ratio=5)
  ax_scatter, ax_top, ax_right = g.ax_joint, g.ax_marg_x, g.ax_marg_y
  sc = ax_scatter.scatter(x, y, s=14, alpha=0.8, linewidths=0, c=phi)
  ax_scatter.set_xlabel("t-SNE 1")
  ax_scatter.set_ylabel("t-SNE 2")
  sns.kdeplot(x=x, ax=ax_top, fill=True, color="#a855f7", bw_adjust=0.6)
  sns.kdeplot(y=y, ax=ax_right, fill=True, color="#f97316", bw_adjust=0.6)
  plt.title(f"t-SNE - Perplexity: {perplexity}")
  plt.show()


In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

features = data_dist.drop(columns=['time'])

phi = data_dih['phi']

# only take every N-th value
stride = 10

features = features.iloc[::stride]
features.reset_index(drop=True, inplace=True)
print(features.shape)

phi = phi.iloc[::stride]
phi.reset_index(drop=True, inplace=True)
print(phi.shape)


perplexities = [20, 40, 60, 80]

for perplexity in perplexities:
  tsne = TSNE(perplexity=perplexity)
  emb = tsne.fit_transform(features)

  x = emb[:,0]
  y = emb[:,1]

  plt.scatter(x, y, s=14, alpha=0.8, linewidths=0, c=data_dih['phi'])
  plt.xlabel("t-SNE 1")
  plt.ylabel("t-SNE 2")
  plt.colorbar()
  plt.show()

  g = sns.JointGrid(height=7, ratio=5)
  ax_scatter, ax_top, ax_right = g.ax_joint, g.ax_marg_x, g.ax_marg_y
  sc = ax_scatter.scatter(x, y, s=14, alpha=0.8, linewidths=0, c=phi)
  ax_scatter.set_xlabel("t-SNE 1")
  ax_scatter.set_ylabel("t-SNE 2")
  sns.kdeplot(x=x, ax=ax_top, fill=True, color="#a855f7", bw_adjust=0.6)
  sns.kdeplot(y=y, ax=ax_right, fill=True, color="#f97316", bw_adjust=0.6)
  plt.title(f"t-SNE - Perplexity: {perplexity}")
  plt.show()
